# QuintetX — AlphaZero Kaggle Resume Trainer

Resume từ checkpoint tại:
- `/kaggle/input/models/hauutogpu/alpha-zero-checkpoint/pytorch/default/1/caro_model_checkpoint.pth`
- `/kaggle/input/models/hauutogpu/alpha-zero-checkpoint/pytorch/default/1/replay_buffer.pkl`
- `/kaggle/input/models/hauutogpu/alpha-zero-checkpoint/pytorch/default/1/alphazero_training_log.csv`

Output lưu tại `/kaggle/working/` — download sau khi session kết thúc.

**Cài đặt trước khi chạy:**
1. Settings → Accelerator → **GPU T4 x2** (hoặc P100)
2. Add Model: `hauutogpu/alpha-zero-checkpoint` vào notebook
3. Run All

In [ ]:
# ============================================================
# CELL 1: Kiểm tra môi trường
# ============================================================
import os, torch

print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

# Kiểm tra input checkpoint
INPUT_BASE = '/kaggle/input/models/hauutogpu/alpha-zero-checkpoint/pytorch/default/3'
INPUT_CHECKPOINT = os.path.join(INPUT_BASE, 'caro_model_checkpoint.pth')
INPUT_BUFFER      = os.path.join(INPUT_BASE, 'replay_buffer.pkl')
INPUT_CSV         = os.path.join(INPUT_BASE, 'alphazero_training_log.csv')

print('\n--- Input files ---')
print(f'  checkpoint : {"✅" if os.path.exists(INPUT_CHECKPOINT) else "❌ KHÔNG TÌM THẤY"}')
print(f'  buffer     : {"✅" if os.path.exists(INPUT_BUFFER) else "❌ KHÔNG TÌM THẤY"}')
print(f'  csv log    : {"✅" if os.path.exists(INPUT_CSV) else "⚠️  Không có — sẽ tạo mới"}')

# Output (read/write)
WORK_DIR = '/kaggle/working'
WORK_CHECKPOINT = os.path.join(WORK_DIR, 'caro_model_checkpoint.pth')
WORK_BUFFER     = os.path.join(WORK_DIR, 'replay_buffer.pkl')
WORK_CSV        = os.path.join(WORK_DIR, 'alphazero_training_log.csv')
print(f'\nOutput dir: {WORK_DIR}')

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random, math, pickle, shutil, csv, time, gc
from collections import deque
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Multi-GPU support
N_GPU = torch.cuda.device_count()
print(f'GPUs available: {N_GPU}')

CSV_HEADER = [
    'iteration', 'total_loss', 'policy_loss', 'value_loss',
    'win_rate_vs_random', 'mean_ep_len', 'new_samples', 'buffer_size',
    'fps', 'elapsed_min', 'learning_rate', 'iter_time_min'
]

In [ ]:
# ============================================================
# CELL 3: Môi trường & luật chơi (40×40, 5-in-a-row)
# ============================================================
BOARD_SIZE = 40
ACTION_SIZE = BOARD_SIZE * BOARD_SIZE  # 1600

def check_win_condition(board, row, col, player):
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
    for dr, dc in directions:
        count = 1
        for sign in (1, -1):
            r, c = row + sign*dr, col + sign*dc
            while 0 <= r < BOARD_SIZE and 0 <= c < BOARD_SIZE and board[r, c] == player:
                count += 1
                r += sign*dr; c += sign*dc
        if count >= 5:
            return True
    return False


class CaroEnvironment:
    def __init__(self):
        self.board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)

    def reset(self):
        self.board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)
        return self.board.copy()

    def step(self, action, player):
        row, col = action // BOARD_SIZE, action % BOARD_SIZE
        if self.board[row, col] != 0:
            raise ValueError(f'Ô ({row},{col}) đã có quân!')
        self.board[row, col] = player
        if check_win_condition(self.board, row, col, player):
            return self.board.copy(), True, player
        if not np.any(self.board == 0):
            return self.board.copy(), True, 0   # Hòa
        return self.board.copy(), False, None

    def get_valid_moves(self):
        return np.where(self.board.flatten() == 0)[0]


def preprocess_state(board, ai_piece):
    """3 channels: [quân mình, quân đối, lượt đi]"""
    opp = 2 if ai_piece == 1 else 1
    t = np.zeros((3, BOARD_SIZE, BOARD_SIZE), dtype=np.float32)
    t[0] = (board == ai_piece).astype(np.float32)
    t[1] = (board == opp).astype(np.float32)
    t[2] = np.ones((BOARD_SIZE, BOARD_SIZE), dtype=np.float32) if ai_piece == 1 else np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=np.float32)
    return t

print('✅ Environment ready')

In [ ]:
# ============================================================
# CELL 4: Kiến trúc mạng (giữ nguyên 100% so với checkpoint)
# ============================================================
class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, 3, padding=1)  # bias=True — khớp checkpoint
        self.bn1   = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, 3, padding=1)  # bias=True — khớp checkpoint
        self.bn2   = nn.BatchNorm2d(num_hidden)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class CaroNet(nn.Module):
    def __init__(self, num_res_blocks=10, num_hidden=128):
        super().__init__()
        # Trunk
        self.conv_start = nn.Conv2d(3, num_hidden, 3, padding=1)  # bias=True — khớp checkpoint
        self.bn_start   = nn.BatchNorm2d(num_hidden)
        self.res_blocks = nn.ModuleList([ResBlock(num_hidden) for _ in range(num_res_blocks)])
        # Policy head
        self.policy_conv = nn.Conv2d(num_hidden, 2, 1)  # bias=True — khớp checkpoint
        self.policy_bn   = nn.BatchNorm2d(2)
        self.policy_fc   = nn.Linear(2 * BOARD_SIZE * BOARD_SIZE, ACTION_SIZE)
        # Value head
        self.value_conv  = nn.Conv2d(num_hidden, 1, 1)  # bias=True — khớp checkpoint
        self.value_bn    = nn.BatchNorm2d(1)
        self.value_fc1   = nn.Linear(BOARD_SIZE * BOARD_SIZE, 256)
        self.value_fc2   = nn.Linear(256, 1)

    def forward(self, x):
        # Trunk
        x = F.relu(self.bn_start(self.conv_start(x)))
        for block in self.res_blocks:
            x = block(x)
        # Policy
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        policy = F.log_softmax(self.policy_fc(p), dim=1)
        # Value
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        value = torch.tanh(self.value_fc2(v))
        return policy, value


n_params = sum(p.numel() for p in CaroNet().parameters())
print(f'CaroNet params: {n_params:,}')

In [ ]:
# ============================================================
# CELL 5: MCTS
# ============================================================
class MCTSNode:
    __slots__ = ['parent', 'children', 'visit_count', 'value_sum', 'prior']

    def __init__(self, prior, parent=None):
        self.parent      = parent
        self.children    = {}
        self.visit_count = 0
        self.value_sum   = 0.0
        self.prior       = prior

    @property
    def value(self):
        return self.value_sum / self.visit_count if self.visit_count > 0 else 0.0

    def select_child(self, c_puct):
        sqrt_n = math.sqrt(self.visit_count)
        best_score, best_action, best_child = -float('inf'), -1, None
        for action, child in self.children.items():
            u = c_puct * child.prior * sqrt_n / (1 + child.visit_count)
            score = child.value + u
            if score > best_score:
                best_score, best_action, best_child = score, action, child
        return best_action, best_child

    def expand(self, probs):
        for action, p in enumerate(probs):
            if p > 0:
                self.children[action] = MCTSNode(prior=p, parent=self)

    def is_expanded(self):
        return len(self.children) > 0


class MCTS:
    def __init__(self, network, c_puct=1.5, n_simulations=800):
        self.network      = network
        self.c_puct       = c_puct
        self.n_simulations = n_simulations

    def _get_net(self):
        """Unwrap DataParallel nếu có"""
        return self.network.module if isinstance(self.network, nn.DataParallel) else self.network

    def get_action_prob(self, state, player, temp=1.0):
        net = self._get_net()
        net.eval()

        root = MCTSNode(prior=0.0)
        root.visit_count = 1

        # Initial expansion
        st = torch.FloatTensor(preprocess_state(state, player)).unsqueeze(0).to(device)
        with torch.no_grad():
            logits, _ = net(st)
        probs = torch.exp(logits).cpu().numpy().flatten()
        valid = (state.flatten() == 0).astype(np.float32)
        probs *= valid
        s = probs.sum()
        probs = probs / s if s > 1e-8 else valid / valid.sum()

        # Dirichlet noise ở root (chỉ khi temp > 0 = training)
        if temp > 0:
            idx = np.where(valid > 0)[0]
            noise = np.zeros(ACTION_SIZE, dtype=np.float32)
            noise[idx] = np.random.dirichlet([0.03] * len(idx))
            probs = 0.75 * probs + 0.25 * noise
            s2 = probs.sum()
            probs /= s2

        root.expand(probs)

        # MCTS simulations
        for _ in range(self.n_simulations):
            self._simulate(root, state.copy(), player)

        # Policy từ visit counts
        actions = sorted(root.children.keys())
        counts  = np.array([root.children[a].visit_count for a in actions], dtype=np.float32)

        if temp == 0:
            out = np.zeros(ACTION_SIZE, dtype=np.float32)
            out[actions[np.argmax(counts)]] = 1.0
            return out

        counts = counts ** (1.0 / temp)
        counts /= counts.sum()
        out = np.zeros(ACTION_SIZE, dtype=np.float32)
        for a, p in zip(actions, counts):
            out[a] = p
        return out

    def _simulate(self, root, state, player):
        node = root
        s    = state
        cur  = player
        path = []

        # Selection
        while node.is_expanded():
            action, node = node.select_child(self.c_puct)
            path.append((node, action))
            s[action // BOARD_SIZE, action % BOARD_SIZE] = cur
            cur = 3 - cur

        # Kiểm tra terminal
        v    = 0.0
        done = False
        if path:
            last_action = path[-1][1]
            last_player = 3 - cur
            r, c = last_action // BOARD_SIZE, last_action % BOARD_SIZE
            if check_win_condition(s, r, c, last_player):
                done, v = True, -1.0   # current player thua
            elif not np.any(s == 0):
                done, v = True, 0.0    # Hòa

        # Expansion + evaluation
        if not done:
            net = self._get_net()
            st  = torch.FloatTensor(preprocess_state(s, cur)).unsqueeze(0).to(device)
            with torch.no_grad():
                logits, val = net(st)
                p = torch.exp(logits).cpu().numpy().flatten()
                v = val.item()
            valid = (s.flatten() == 0).astype(np.float32)
            p *= valid
            ps = p.sum()
            p  = p / ps if ps > 1e-8 else valid / valid.sum()
            node.expand(p)

        # Backpropagation
        node.visit_count += 1
        node.value_sum   += v
        v = -v
        for n, _ in reversed(path[:-1]):
            n.visit_count += 1
            n.value_sum   += v
            v = -v

print('✅ MCTS ready')

In [ ]:
# ============================================================
# CELL 6: Self-play, Train, Eval
# ============================================================
def execute_episode(network, mcts, env, ep_num):
    """Chạy 1 ván self-play. Trả về (data, move_count, duration_sec)"""
    examples = []
    state    = env.reset()
    player   = 1
    t0       = time.time()
    max_moves = 200  # bảng 40×40 hiếm khi dài hơn

    for step in range(max_moves):
        temp   = 1.0 if step < 30 else 0.1
        pi     = mcts.get_action_prob(state, player, temp=temp)
        examples.append([preprocess_state(state, player), pi, player])
        action = int(np.random.choice(ACTION_SIZE, p=pi))
        state, done, winner = env.step(action, player)
        if done:
            dur = time.time() - t0
            if winner == 0:
                z = 0.0
                data = [(x[0], x[1], 0.0) for x in examples]
            else:
                data = [(x[0], x[1],
                         1.0 if x[2] == winner else -1.0)
                        for x in examples]
            return data, step + 1, dur
        player = 3 - player

    dur = time.time() - t0
    return [(x[0], x[1], 0.0) for x in examples], max_moves, dur


class CaroDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        state, pi, z = self.data[i]
        return (torch.FloatTensor(state),
                torch.FloatTensor(pi),
                torch.FloatTensor([z]))


def train_network(network, optimizer, data, batch_size=256, epochs=5):
    """Trả về (total_loss, value_loss, policy_loss) của epoch cuối"""
    loader = DataLoader(
        CaroDataset(data),
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=True
    )
    network.train()
    tl = vl = pl = 0.0

    for ep in range(epochs):
        et = ev = ep_ = n = 0
        for states, pis, zs in loader:
            states = states.to(device, non_blocking=True)
            pis    = pis.to(device, non_blocking=True)
            zs     = zs.to(device, non_blocking=True).squeeze(-1)

            optimizer.zero_grad()
            log_p, v = network(states)
            v  = v.squeeze(-1)
            v_loss  = F.mse_loss(v, zs)
            p_loss  = -(pis * log_p).sum(dim=1).mean()
            loss    = v_loss + p_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
            optimizer.step()

            et += loss.item(); ev += v_loss.item(); ep_ += p_loss.item(); n += 1

        tl, vl, pl = et/n, ev/n, ep_/n
        print(f'   Epoch {ep+1}/{epochs} | total={tl:.4f} | value={vl:.4f} | policy={pl:.4f}')

    return tl, vl, pl


def eval_vs_random(network, env, n_games=20):
    """Đánh giá greedy policy (không MCTS) vs random. Nhanh ~30s"""
    net = network.module if isinstance(network, nn.DataParallel) else network
    net.eval()
    wins = 0
    for g in range(n_games):
        state        = env.reset()
        model_player = 1 if g % 2 == 0 else 2
        cur          = 1
        for _ in range(400):
            valid = np.where(state.flatten() == 0)[0]
            if len(valid) == 0: break
            if cur == model_player:
                st = torch.FloatTensor(preprocess_state(state, cur)).unsqueeze(0).to(device)
                with torch.no_grad():
                    lp, _ = net(st)
                    p = torch.exp(lp).cpu().numpy().flatten()
                mask = np.zeros(ACTION_SIZE, dtype=np.float32)
                mask[valid] = 1.0
                p *= mask
                ps = p.sum()
                p  = p / ps if ps > 1e-8 else mask / mask.sum()
                action = int(np.argmax(p))
            else:
                action = int(np.random.choice(valid))
            try:
                state, done, winner = env.step(action, cur)
            except ValueError:
                break
            if done:
                if winner == model_player: wins += 1
                break
            cur = 3 - cur
    wr = wins / n_games
    print(f'   📊 Eval vs Random: {wins}/{n_games} = {wr:.1%}')
    return wr


print('✅ Self-play / Train / Eval ready')

In [ ]:
# ============================================================
# CELL 7: Save / Load / CSV
# ============================================================
def init_csv():
    """Copy CSV từ input nếu có, ngược lại tạo mới với header."""
    if not os.path.exists(WORK_CSV):
        if os.path.exists(INPUT_CSV):
            shutil.copy(INPUT_CSV, WORK_CSV)
            print(f'📋 Resume CSV từ input ({INPUT_CSV})')
        else:
            with open(WORK_CSV, 'w', newline='') as f:
                csv.writer(f).writerow(CSV_HEADER)
            print(f'📋 Tạo CSV mới: {WORK_CSV}')
    else:
        print(f'📋 CSV đã tồn tại, tiếp tục append')


def log_row(iteration, total_loss, policy_loss, value_loss,
            win_rate, mean_ep_len, new_samples, buffer_size,
            fps, elapsed_min, lr, iter_time_min):
    row = [
        iteration,
        round(total_loss,   6), round(policy_loss,  6), round(value_loss,  6),
        round(win_rate,     4), round(mean_ep_len,   1),
        new_samples, buffer_size,
        round(fps,          1), round(elapsed_min,   2),
        lr,                     round(iter_time_min, 2)
    ]
    with open(WORK_CSV, 'a', newline='') as f:
        csv.writer(f).writerow(row)
    print(f'  📝 iter={iteration} | loss={total_loss:.4f} | '
          f'win={win_rate:.1%} | fps={fps:.1f} | {elapsed_min:.1f}min')


def save_checkpoint(network, optimizer, iteration, buffer):
    """Lưu checkpoint vào /kaggle/working/"""
    net_state = (network.module.state_dict()
                 if isinstance(network, nn.DataParallel)
                 else network.state_dict())
    torch.save({
        'iteration':       iteration,
        'model_state':     net_state,
        'optimizer_state': optimizer.state_dict()
    }, WORK_CHECKPOINT)
    # Buffer: lưu mỗi 5 iter vì nặng
    if iteration % 5 == 0:
        with open(WORK_BUFFER, 'wb') as f:
            pickle.dump(list(buffer), f)
        print(f'💾 iter={iteration} | checkpoint + buffer + CSV saved')
    else:
        print(f'💾 iter={iteration} | checkpoint + CSV saved')


def load_checkpoint(network, optimizer):
    """
    Thứ tự ưu tiên:
      1. /kaggle/working/ (nếu đang resume trong cùng session)
      2. /kaggle/input/   (checkpoint cũ từ Kaggle model)
    """
    src = None
    if os.path.exists(WORK_CHECKPOINT):
        src = WORK_CHECKPOINT
        print('📥 Load checkpoint từ WORK (/kaggle/working/)')
    elif os.path.exists(INPUT_CHECKPOINT):
        src = INPUT_CHECKPOINT
        print(f'📥 Load checkpoint từ INPUT ({INPUT_CHECKPOINT})')

    if src:
        ckpt = torch.load(src, map_location=device)
        net_state = ckpt.get('model_state', ckpt)  # tương thích cả 2 format
        net = network.module if isinstance(network, nn.DataParallel) else network
        net.load_state_dict(net_state)
        if 'optimizer_state' in ckpt:
            try:
                optimizer.load_state_dict(ckpt['optimizer_state'])
            except Exception as e:
                print(f'⚠️  Không load được optimizer state: {e} — reset optimizer')
        start = ckpt.get('iteration', -1) + 1
        print(f'✅ Resume từ iteration {start}')
        return start

    print('⚠️  Không tìm thấy checkpoint — bắt đầu từ iteration 0')
    return 0


def load_buffer(maxlen=200_000):
    """
    Thứ tự ưu tiên:
      1. /kaggle/working/replay_buffer.pkl
      2. /kaggle/input/.../replay_buffer.pkl
    """
    for src, label in [(WORK_BUFFER, 'WORK'), (INPUT_BUFFER, 'INPUT')]:
        if os.path.exists(src):
            try:
                print(f'📥 Load buffer từ {label} ({src})...')
                with open(src, 'rb') as f:
                    data = pickle.load(f)
                buf = deque(data, maxlen=maxlen)
                print(f'✅ Buffer: {len(buf):,} samples')
                return buf
            except Exception as e:
                print(f'⚠️  Lỗi load buffer từ {label}: {e}')
    print('⚠️  Không tìm thấy buffer — bắt đầu trống')
    return deque(maxlen=maxlen)


print('✅ Save / Load / CSV ready')

In [ ]:
# ============================================================
# CELL 8: CONFIG — chỉnh tại đây nếu cần
# ============================================================
CFG = {
    # Kiến trúc
    'num_res_blocks': 10,
    'num_hidden':     128,

    # MCTS
    'n_simulations':  800,
    'c_puct':         1.5,

    # Self-play
    'episodes_per_iter': 10,

    # Training
    'batch_size':       256,
    'epochs_per_iter':  5,
    'lr':               1e-4,     # fine-tune LR (đã qua pre-train)
    'weight_decay':     1e-4,
    'max_grad_norm':    1.0,

    # Buffer
    'buffer_maxlen':    200_000,
    'train_sample':     50_000,   # số sample dùng mỗi iter
    'min_buffer':       1_000,    # bắt đầu train khi buffer >= này

    # Loop
    'max_iterations':   200,
    'eval_games':       20,
}

print('Config:')
for k, v in CFG.items():
    print(f'  {k:25s}: {v}')

In [ ]:
# ============================================================
# CELL 9: MAIN — Resume train
# ============================================================
def main():
    print(f'🚀 QuintetX AlphaZero | Kaggle Resume | device={device}')

    # --- Khởi tạo ---
    env     = CaroEnvironment()
    network = CaroNet(
        num_res_blocks=CFG['num_res_blocks'],
        num_hidden=CFG['num_hidden']
    ).to(device)

    # Multi-GPU nếu có
    if N_GPU > 1:
        network = nn.DataParallel(network)
        print(f'  DataParallel trên {N_GPU} GPU')

    optimizer = optim.Adam(
        network.parameters(),
        lr=CFG['lr'],
        weight_decay=CFG['weight_decay']
    )

    # --- Load checkpoint & buffer ---
    start_iter = load_checkpoint(network, optimizer)
    buffer     = load_buffer(maxlen=CFG['buffer_maxlen'])
    init_csv()

    # Cập nhật LR (optimizer state từ checkpoint có thể dùng LR cũ)
    for pg in optimizer.param_groups:
        pg['lr'] = CFG['lr']
    print(f'  LR set to {CFG["lr"]}')

    mcts     = MCTS(network, c_puct=CFG['c_puct'], n_simulations=CFG['n_simulations'])
    t_global = time.time()

    print(f'\n[RL] Bắt đầu từ iteration {start_iter}')
    print(f'     Buffer hiện tại: {len(buffer):,} samples')
    print(f'     Mỗi iteration ước tính ~30–60 phút trên T4 (10 eps × 800 sims)')
    print('=' * 60)

    for i in range(start_iter, CFG['max_iterations']):
        print(f"\n{'='*60}")
        print(f"ITERATION {i}  |  buffer={len(buffer):,}")
        print(f"{'='*60}")
        t_iter = time.time()

        # ---- Self-play ----
        total_moves  = 0
        total_time   = 0.0
        new_samples  = 0

        for ep in range(CFG['episodes_per_iter']):
            data, moves, dur = execute_episode(network, mcts, env, ep + 1)
            buffer.extend(data)
            new_samples  += len(data)
            total_moves  += moves
            total_time   += dur
            print(f'  ep {ep+1:2d}/{CFG["episodes_per_iter"]} | '
                  f'moves={moves:3d} | {dur/60:.1f}min | buffer={len(buffer):,}')

        mean_ep_len = total_moves / CFG['episodes_per_iter']
        fps         = total_moves / total_time if total_time > 0 else 0.0

        # ---- Train ----
        if len(buffer) < CFG['min_buffer']:
            print(f'⏭️  Buffer {len(buffer):,} < min {CFG["min_buffer"]:,} — bỏ qua train')
            continue

        n_sample   = min(len(buffer), CFG['train_sample'])
        train_data = random.sample(list(buffer), n_sample)
        print(f'\n  🏋️  Train trên {n_sample:,} samples...')
        tl, vl, pl = train_network(
            network, optimizer, train_data,
            batch_size=CFG['batch_size'],
            epochs=CFG['epochs_per_iter']
        )

        # ---- Eval ----
        print(f'\n  📊 Eval vs Random ({CFG["eval_games"]} games)...')
        win_rate = eval_vs_random(network, env, n_games=CFG['eval_games'])

        # ---- Log & Save ----
        elapsed_min   = (time.time() - t_global) / 60
        iter_time_min = (time.time() - t_iter)   / 60
        log_row(
            i, tl, pl, vl, win_rate, mean_ep_len,
            new_samples, len(buffer),
            fps, elapsed_min, CFG['lr'], iter_time_min
        )
        save_checkpoint(network, optimizer, i, buffer)

        # Dọn VRAM
        gc.collect()
        torch.cuda.empty_cache()

    print('\n🏁 Training hoàn tất!')
    print(f'   Output files trong /kaggle/working/')
    for f in [WORK_CHECKPOINT, WORK_BUFFER, WORK_CSV]:
        exists = os.path.exists(f)
        size   = os.path.getsize(f) / 1e6 if exists else 0
        print(f'   {"✅" if exists else "❌"} {f}  ({size:.1f} MB)')


main()

In [ ]:
# ============================================================
# CELL 10 (tuỳ chọn): Xem CSV log ngay trong notebook
# ============================================================
import pandas as pd

if os.path.exists(WORK_CSV):
    df = pd.read_csv(WORK_CSV)
    print(f'Total iterations logged: {len(df)}')
    display(df.tail(20))
else:
    print('Chưa có CSV')

In [ ]:
# ============================================================
# CELL 11 (tuỳ chọn): Plot training curve
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd

if os.path.exists(WORK_CSV):
    df = pd.read_csv(WORK_CSV)
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle('QuintetX AlphaZero — Training Progress', fontsize=14, fontweight='bold')

    axes[0,0].plot(df['iteration'], df['total_loss'],  color='steelblue')
    axes[0,0].set_title('Total Loss');  axes[0,0].set_xlabel('Iteration')

    axes[0,1].plot(df['iteration'], df['policy_loss'], color='darkorange')
    axes[0,1].set_title('Policy Loss'); axes[0,1].set_xlabel('Iteration')

    axes[1,0].plot(df['iteration'], df['value_loss'],  color='green')
    axes[1,0].set_title('Value Loss');  axes[1,0].set_xlabel('Iteration')

    axes[1,1].plot(df['iteration'], df['win_rate_vs_random'], color='crimson')
    axes[1,1].axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
    axes[1,1].set_title('Win Rate vs Random'); axes[1,1].set_xlabel('Iteration')
    axes[1,1].set_ylim(0, 1.05)

    for ax in axes.flat:
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out_fig = os.path.join(WORK_DIR, 'alphazero_training_curve.png')
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {out_fig}')
else:
    print('Chưa có CSV để plot')